In [1]:
from langchain_chroma import Chroma
# Query
def query(vector_store: Chroma, question: str, project: str = None, k: int = 5):
    """Search the vector store and print results."""
    search_kwargs = {"k": k}
    if project:
        search_kwargs["filter"] = {"project": project}
 
    results = vector_store.similarity_search(question, **search_kwargs)
 
    print(f"\nQuery: {question}")
    print(f"Results ({len(results)}):\n")
    for i, doc in enumerate(results, 1):
        print(f"  {i}. [{doc.metadata['source']}]")
        print(f"     {doc.metadata.get('heading_trail', '')}")
        print(f"     {doc.page_content[:150]}...")
        print(f"{'-'*50}")
 
    return results


In [12]:
# 
from langchain_openai import OpenAIEmbeddings
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model = embedding_model)
vector_store = Chroma(
    collection_name = "nextjs",
    embedding_function=embeddings,
    persist_directory="../vectorstore"
)

query(vector_store, "what is next js" )



Query: what is next js
Results (5):

  1. [index.mdx]
     What is Next.js?
     What is Next.js?

## What is Next.js?  
Next.js is a React framework for building full-stack web applications. You use React Components to build user ...
--------------------------------------------------
  2. [supported-browsers.mdx]
     JavaScript Language Features
     JavaScript Language Features

## JavaScript Language Features  
Next.js allows you to use the latest JavaScript features out of the box. In addition t...
--------------------------------------------------
  3. [supported-browsers.mdx]
     JavaScript Language Features > TypeScript Features
     JavaScript Language Features > TypeScript Features

### TypeScript Features  
Next.js has built-in TypeScript support. [Learn more here](/docs/pages/a...
--------------------------------------------------
  4. [single-page-applications.mdx]
     Why use Next.js for SPAs?
     Why use Next.js for SPAs?

## Why use Next.js for SPAs?  
Next.js can a

[Document(id='fca3f847b93fdd39ce2a45bd36f65a53', metadata={'heading_trail': 'What is Next.js?', 'source': 'index.mdx', 'project': 'nextjs'}, page_content="What is Next.js?\n\n## What is Next.js?  \nNext.js is a React framework for building full-stack web applications. You use React Components to build user interfaces, and Next.js for additional features and optimizations.  \nIt also automatically configures lower-level tools like bundlers and compilers. You can instead focus on building your product and shipping quickly.  \nWhether you're an individual developer or part of a larger team, Next.js can help you build interactive, dynamic, and fast React applications."),
 Document(id='3586b9b1944d8c0f9e60204cb6c2008c', metadata={'source': 'supported-browsers.mdx', 'heading_trail': 'JavaScript Language Features', 'project': 'nextjs'}, page_content='JavaScript Language Features\n\n## JavaScript Language Features  \nNext.js allows you to use the latest JavaScript features out of the box. In a

In [11]:
# Check if ChromaDB has any data
import chromadb

client = chromadb.PersistentClient(path="../vectorstore")
collection = client.get_collection("nextjs")
print(f"Total chunks stored: {collection.count()}")

Total chunks stored: 3164


In [8]:
from __future__ import annotations
 
import re
from dataclasses import dataclass, field
from typing import Callable, List, Tuple
 
from langchain_core.documents import Document


@dataclass
class RetrievalEvalSample:
    """Single evaluation sample."""
 
    query: str
    expected_keywords: List[str]           # must appear in at least one retrieved doc
    expected_sources: List[str]            # substring matches against doc metadata["source"]
    relevant_doc_identifiers: List[str]    # substrings that mark a doc as "relevant" (for MRR)


EVAL_DATASET_EXTENDED: List[RetrievalEvalSample] = [

    # ── Sequence-to-Sequence / T5 ──────────────────────────────────────────
    RetrievalEvalSample(
        query="How to use T5 for text-to-text tasks like summarization and translation?",
        expected_keywords=["t5", "seq2seq", "summarization", "translation", "encoder_decoder"],
        expected_sources=["t5", "model_doc/t5", "summarization", "translation"],
        relevant_doc_identifiers=["t5", "seq2seq"],
    ),

    # ── Image classification / ViT ─────────────────────────────────────────
    RetrievalEvalSample(
        query="How to fine-tune a Vision Transformer ViT for image classification?",
        expected_keywords=["vit", "image_classification", "AutoModelForImageClassification", "patch"],
        expected_sources=["vit", "image_classification", "model_doc/vit"],
        relevant_doc_identifiers=["vit", "image_classification"],
    ),

    # ── Object detection / DETR ────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to perform object detection using DETR in transformers?",
        expected_keywords=["detr", "object_detection", "bounding_box", "coco"],
        expected_sources=["detr", "object_detection", "model_doc/detr"],
        relevant_doc_identifiers=["detr", "object_detection"],
    ),

    # ── Speech / Whisper ───────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to transcribe audio to text using the Whisper model?",
        expected_keywords=["whisper", "transcribe", "audio", "speech_recognition", "AutoModelForSpeechSeq2Seq"],
        expected_sources=["whisper", "model_doc/whisper", "automatic_speech_recognition"],
        relevant_doc_identifiers=["whisper", "speech_recognition", "transcribe"],
    ),

    # ── Masked Language Modeling ────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to train a masked language model like BERT for pretraining?",
        expected_keywords=["masked", "mlm", "bert", "AutoModelForMaskedLM", "DataCollatorForLanguageModeling"],
        expected_sources=["language_modeling", "mlm", "bert"],
        relevant_doc_identifiers=["masked", "mlm"],
    ),

    # ── Named Entity Recognition ───────────────────────────────────────────
    RetrievalEvalSample(
        query="How to fine-tune a model for named entity recognition NER token classification?",
        expected_keywords=["ner", "token_classification", "AutoModelForTokenClassification", "label", "B-PER"],
        expected_sources=["token_classification", "ner"],
        relevant_doc_identifiers=["token_classification", "ner"],
    ),

    # ── Question Answering ─────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to build an extractive question answering system with transformers?",
        expected_keywords=["question_answering", "AutoModelForQuestionAnswering", "start_logits", "end_logits", "context"],
        expected_sources=["question_answering", "qa"],
        relevant_doc_identifiers=["question_answering", "qa"],
    ),

    # ── Semantic Similarity / Sentence Transformers ────────────────────────
    RetrievalEvalSample(
        query="How to compute sentence embeddings for semantic similarity search?",
        expected_keywords=["embedding", "sentence", "similarity", "cosine", "pooling"],
        expected_sources=["sentence_transformers", "semantic_similarity"],
        relevant_doc_identifiers=["embedding", "sentence", "similarity"],
    ),

    # ── KV Cache / Efficient Inference ─────────────────────────────────────
    RetrievalEvalSample(
        query="What is KV cache and how does it speed up autoregressive generation?",
        expected_keywords=["kv_cache", "past_key_values", "cache", "autoregressive"],
        expected_sources=["generation", "kv_cache", "llm_optims"],
        relevant_doc_identifiers=["kv_cache", "past_key_values", "cache"],
    ),

    # ── Gradient Checkpointing ─────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to enable gradient checkpointing to reduce memory during training?",
        expected_keywords=["gradient_checkpointing", "memory", "checkpoint", "recompute"],
        expected_sources=["perf_train_gpu_one", "gradient_checkpointing", "training"],
        relevant_doc_identifiers=["gradient_checkpointing", "checkpoint"],
    ),

    # ── Mixed Precision / FP16 / BF16 ─────────────────────────────────────
    RetrievalEvalSample(
        query="How to use mixed precision FP16 or BF16 training for faster performance?",
        expected_keywords=["fp16", "bf16", "mixed_precision", "half_precision", "amp"],
        expected_sources=["perf_train_gpu_one", "mixed_precision", "training"],
        relevant_doc_identifiers=["fp16", "bf16", "mixed_precision"],
    ),

    # ── DeepSpeed Integration ──────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to integrate DeepSpeed ZeRO for large-scale model training?",
        expected_keywords=["deepspeed", "zero", "stage2", "stage3", "offload"],
        expected_sources=["deepspeed", "main_classes/deepspeed", "perf_train_gpu_many"],
        relevant_doc_identifiers=["deepspeed", "zero"],
    ),

    # ── GPTQ Quantization ─────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to load a GPTQ quantized model for efficient inference?",
        expected_keywords=["gptq", "quantization", "auto_gptq", "bits", "exllama"],
        expected_sources=["quantization", "gptq", "main_classes/quantization"],
        relevant_doc_identifiers=["gptq", "quantization"],
    ),

    # ── AWQ Quantization ──────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to use AWQ activation-aware weight quantization in transformers?",
        expected_keywords=["awq", "quantization", "activation", "weight"],
        expected_sources=["quantization", "awq", "main_classes/quantization"],
        relevant_doc_identifiers=["awq", "quantization"],
    ),

    # ── Callbacks / Logging ────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to use callbacks for logging training metrics to wandb or tensorboard?",
        expected_keywords=["callback", "TrainerCallback", "wandb", "tensorboard", "logging"],
        expected_sources=["callback", "trainer", "logging"],
        relevant_doc_identifiers=["callback", "logging", "wandb"],
    ),

    # ── Custom Model / Modeling Code ───────────────────────────────────────
    RetrievalEvalSample(
        query="How to write a custom model class inheriting from PreTrainedModel?",
        expected_keywords=["PreTrainedModel", "config", "forward", "custom", "modeling"],
        expected_sources=["custom_models", "create_a_model", "model_doc"],
        relevant_doc_identifiers=["PreTrainedModel", "custom_model"],
    ),

    # ── Feature Extraction ─────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to extract hidden state features from a pretrained model?",
        expected_keywords=["hidden_states", "output_hidden_states", "last_hidden_state", "feature_extraction"],
        expected_sources=["feature_extraction", "model_outputs"],
        relevant_doc_identifiers=["hidden_states", "feature_extraction"],
    ),

    # ── Multi-modal / CLIP ─────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How does CLIP work for matching images and text using contrastive learning?",
        expected_keywords=["clip", "image", "text", "contrastive", "CLIPModel"],
        expected_sources=["clip", "model_doc/clip"],
        relevant_doc_identifiers=["clip", "contrastive"],
    ),

    # ── LLaMA / Llama ─────────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to load and run inference with Llama 2 or Llama 3 models?",
        expected_keywords=["llama", "LlamaForCausalLM", "meta", "from_pretrained"],
        expected_sources=["llama", "llama2", "model_doc/llama"],
        relevant_doc_identifiers=["llama"],
    ),

    # ── Mistral ────────────────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to use Mistral 7B model with sliding window attention?",
        expected_keywords=["mistral", "sliding_window", "attention", "MistralForCausalLM"],
        expected_sources=["mistral", "model_doc/mistral"],
        relevant_doc_identifiers=["mistral", "sliding_window"],
    ),

    # ── Chat Templates ─────────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to format messages using chat templates and apply_chat_template?",
        expected_keywords=["chat_template", "apply_chat_template", "messages", "jinja"],
        expected_sources=["chat_templating", "chat_template"],
        relevant_doc_identifiers=["chat_template", "apply_chat_template"],
    ),

    # ── Streaming Generation ───────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to stream token-by-token output during text generation?",
        expected_keywords=["streamer", "TextStreamer", "TextIteratorStreamer", "streaming"],
        expected_sources=["generation_strategies", "streaming", "text_generation"],
        relevant_doc_identifiers=["streamer", "streaming"],
    ),

    # ── Stopping Criteria ──────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to define custom stopping criteria for text generation?",
        expected_keywords=["StoppingCriteria", "stopping_criteria", "MaxLengthCriteria", "eos_token"],
        expected_sources=["generation_strategies", "stopping_criteria", "generation"],
        relevant_doc_identifiers=["stopping_criteria", "StoppingCriteria"],
    ),

    # ── Evaluation / Metrics ───────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to evaluate a model with metrics like accuracy F1 and BLEU using evaluate library?",
        expected_keywords=["evaluate", "metric", "accuracy", "f1", "bleu", "compute"],
        expected_sources=["evaluate", "training", "tasks"],
        relevant_doc_identifiers=["evaluate", "metric"],
    ),

    # ── Configuration / PretrainedConfig ───────────────────────────────────
    RetrievalEvalSample(
        query="How to modify model configuration like hidden size and number of layers?",
        expected_keywords=["config", "PretrainedConfig", "hidden_size", "num_hidden_layers", "from_pretrained"],
        expected_sources=["configuration", "config", "create_a_model"],
        relevant_doc_identifiers=["config", "PretrainedConfig"],
    ),

    # ── Image Processor / Feature Extractor ────────────────────────────────
    RetrievalEvalSample(
        query="How to preprocess images with AutoImageProcessor for vision models?",
        expected_keywords=["AutoImageProcessor", "image_processor", "resize", "normalize", "pixel_values"],
        expected_sources=["image_processor", "preprocessing", "model_doc/auto"],
        relevant_doc_identifiers=["image_processor", "AutoImageProcessor"],
    ),

    # ── Zero-shot Classification ───────────────────────────────────────────
    RetrievalEvalSample(
        query="How to do zero-shot text classification without training on specific labels?",
        expected_keywords=["zero_shot", "classification", "candidate_labels", "hypothesis", "nli"],
        expected_sources=["zero_shot", "pipeline", "zeroshot"],
        relevant_doc_identifiers=["zero_shot", "candidate_labels"],
    ),

    # ── Text Summarization ─────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to fine-tune a model for abstractive text summarization?",
        expected_keywords=["summarization", "Seq2SeqTrainer", "rouge", "AutoModelForSeq2SeqLM"],
        expected_sources=["summarization", "tasks/summarization"],
        relevant_doc_identifiers=["summarization"],
    ),

    # ── Translation ────────────────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to fine-tune MarianMT or mBART for machine translation?",
        expected_keywords=["translation", "marian", "mbart", "source_lang", "target_lang"],
        expected_sources=["translation", "marian", "mbart"],
        relevant_doc_identifiers=["translation", "marian", "mbart"],
    ),

    # ── Model Parallelism / Tensor Parallel ────────────────────────────────
    RetrievalEvalSample(
        query="How to split a large model across multiple GPUs with device_map auto?",
        expected_keywords=["device_map", "auto", "offload", "disk_offload", "model_parallelism"],
        expected_sources=["big_models", "accelerate", "perf_infer_gpu_many"],
        relevant_doc_identifiers=["device_map", "model_parallelism", "offload"],
    ),

    # ── Tokenizer Training / BPE ───────────────────────────────────────────
    RetrievalEvalSample(
        query="How to train a new BPE tokenizer from scratch on a custom corpus?",
        expected_keywords=["bpe", "train", "tokenizer", "vocab_size", "ByteLevelBPETokenizer"],
        expected_sources=["tokenizer_summary", "tokenizers", "training_tokenizer"],
        relevant_doc_identifiers=["bpe", "train", "tokenizer"],
    ),

    # ── Retrieval Augmented Generation ─────────────────────────────────────
    RetrievalEvalSample(
        query="How to implement RAG retrieval augmented generation with transformers?",
        expected_keywords=["rag", "retrieval", "RagTokenForGeneration", "knowledge", "dense_passage"],
        expected_sources=["rag", "model_doc/rag"],
        relevant_doc_identifiers=["rag", "retrieval_augmented"],
    ),

    # ── GGUF / llama.cpp Format ────────────────────────────────────────────
    RetrievalEvalSample(
        query="How to load GGUF quantized model files in HuggingFace transformers?",
        expected_keywords=["gguf", "ggml", "llama_cpp", "from_pretrained", "quantized"],
        expected_sources=["gguf", "quantization", "llama"],
        relevant_doc_identifiers=["gguf"],
    ),

    # ── Safetensors ────────────────────────────────────────────────────────
    RetrievalEvalSample(
        query="What is safetensors format and how to save and load models with it?",
        expected_keywords=["safetensors", "save_pretrained", "safe_serialization", "load", "torch"],
        expected_sources=["safetensors", "serialization", "big_models"],
        relevant_doc_identifiers=["safetensors"],
    ),

    # ── Instruction Tuning / SFT ───────────────────────────────────────────
    RetrievalEvalSample(
        query="How to instruction-tune or supervised fine-tune an LLM with chat data?",
        expected_keywords=["instruction", "sft", "fine_tune", "chat", "supervised"],
        expected_sources=["sft", "trl", "instruction_tuning", "chat_templating"],
        relevant_doc_identifiers=["sft", "instruction", "fine_tune"],
    ),
]
 
 

In [5]:
from typing import List, Tuple
from langchain_core.documents import Document
from langchain_chroma import Chroma




def retrieve_with_similarity_score(
    vectorstore: Chroma,
    query: str,
    k: int = 10,
) -> List[Tuple[Document, float]]:
    """
    Retrieve documents from Chroma with L2 distances converted to
    similarity scores in [0, 1], sorted highest-similarity-first.

    Chroma's default distance is L2 (squared Euclidean).
    Conversion:  similarity = 1 / (1 + l2_distance)
      - identical vectors  → distance = 0  → similarity = 1.0
      - distant vectors    → distance → ∞  → similarity → 0.0

    Returns:
        List of (Document, similarity_score) tuples, descending by score.
    """
    # similarity_search_with_score returns (Document, L2 distance) pairs
    results: List[Tuple[Document, float]] = (
        vectorstore.similarity_search_with_score(query, k=k)
    )

    # Convert L2 distance → similarity score
    scored_results: List[Tuple[Document, float]] = [
        (doc, 1.0 / (1.0 + l2_distance))
        for doc, l2_distance in results
    ]

    # Sort by similarity score descending (most relevant first)
    scored_results.sort(key=lambda pair: pair[1], reverse=True)

    return scored_results

In [6]:
from typing import List, Tuple, Dict, Any
from langchain_core.documents import Document


def evaluate_retrieval(
    queries: List[Dict[str, Any]],
    retrieve_fn,
    k: int = 5,
) -> Dict[str, Any]:
    """
    Evaluate retrieval on keyword match, source match, and MRR in one pass.

    Parameters
    ----------
    queries : list of dicts, each with:
        - "query": str
        - "expected_keywords": list[str]
        - "expected_sources": list[str]
        - "relevant_doc_identifiers": list[str]   (substrings marking a doc as relevant for MRR)
    retrieve_fn :
        callable(query: str, k: int) -> list[(Document, float)]
        Must return results sorted by descending similarity score.
    k : int
        Number of docs to retrieve per query.

    Returns
    -------
    dict with avg_keyword_recall, avg_source_match_rate, mrr,
    hit_rate_at_1, hit_rate_at_3, hit_rate_at_k, and per_query breakdown.
    """

    per_query = []

    for sample in queries:
        # handle both dict and dataclass
        g = sample.get if isinstance(sample, dict) else sample.__dict__.get

        query = g("query")
        expected_kw = g("expected_keywords", [])
        expected_src = g("expected_sources", [])
        relevance_ids = g("relevant_doc_identifiers", [])

        results: List[Tuple[Document, float]] = retrieve_fn(query, k)

        # --- keyword recall ---
        all_text = " ".join(doc.page_content for doc, _ in results).lower().replace("-", "").replace("_", "")
        matched_kw = [kw for kw in expected_kw if kw.lower().replace("-", "").replace("_", "") in all_text]
        missed_kw = [kw for kw in expected_kw if kw not in matched_kw]
        kw_recall = len(matched_kw) / len(expected_kw) if expected_kw else 1.0

        # --- source match ---
        sources = [str(doc.metadata.get("source", "")).lower() for doc, _ in results]
        matched_src = [s for s in expected_src if any(s.lower() in src for src in sources)]
        missed_src = [s for s in expected_src if s not in matched_src]
        src_rate = len(matched_src) / len(expected_src) if expected_src else 1.0

        # --- MRR (reciprocal rank of first relevant doc) ---
        rr = 0.0
        first_rank = None
        for rank, (doc, _) in enumerate(results, start=1):
            combined = (doc.page_content + " " + str(doc.metadata.get("source", ""))).lower()
            if any(rid.lower() in combined for rid in relevance_ids):
                rr = 1.0 / rank
                first_rank = rank
                break

        per_query.append({
            "query": query,
            "keyword_recall": round(kw_recall, 4),
            "source_match_rate": round(src_rate, 4),
            "reciprocal_rank": round(rr, 4),
            "first_relevant_rank": first_rank,
            "matched_keywords": matched_kw,
            "missed_keywords": missed_kw,
            "matched_sources": matched_src,
            "missed_sources": missed_src,
        })

    n = len(per_query)
    return {
        "num_samples": n,
        "avg_keyword_recall": round(sum(q["keyword_recall"] for q in per_query) / n, 4),
        "avg_source_match_rate": round(sum(q["source_match_rate"] for q in per_query) / n, 4),
        "mrr": round(sum(q["reciprocal_rank"] for q in per_query) / n, 4),
        "hit_rate_at_1": round(sum(1 for q in per_query if q["first_relevant_rank"] == 1) / n, 4),
        "hit_rate_at_3": round(sum(1 for q in per_query if q["first_relevant_rank"] is not None and q["first_relevant_rank"] <= 3) / n, 4),
        "hit_rate_at_k": round(sum(1 for q in per_query if q["first_relevant_rank"] is not None) / n, 4),
        "per_query": per_query,
    }

In [10]:
stats = evaluate_retrieval(
    queries=EVAL_DATASET_EXTENDED,
    retrieve_fn=lambda query, k: retrieve_with_similarity_score(vector_store, query, k),
    k=5,
)

print(f"Keyword Recall : {stats['avg_keyword_recall']:.2%}")
print(f"Source Match    : {stats['avg_source_match_rate']:.2%}")
print(f"MRR            : {stats['mrr']:.4f}")
print(f"Hit@1          : {stats['hit_rate_at_1']:.2%}")
print(f"Hit@3          : {stats['hit_rate_at_3']:.2%}")

# drill into failures
for q in stats["per_query"]:
    if q["missed_keywords"]:
        print(f"\n{q['query'][:60]}...")
        print(f"  missed kw: {q['missed_keywords']}")

Keyword Recall : 82.57%
Source Match    : 37.38%
MRR            : 0.9500
Hit@1          : 91.43%
Hit@3          : 97.14%

How to perform object detection using DETR in transformers?...
  missed kw: ['bounding_box']

How to transcribe audio to text using the Whisper model?...
  missed kw: ['AutoModelForSpeechSeq2Seq']

How to train a masked language model like BERT for pretraini...
  missed kw: ['DataCollatorForLanguageModeling']

How to fine-tune a model for named entity recognition NER to...
  missed kw: ['B-PER']

How to compute sentence embeddings for semantic similarity s...
  missed kw: ['pooling']

What is KV cache and how does it speed up autoregressive gen...
  missed kw: ['past_key_values']

How to use mixed precision FP16 or BF16 training for faster ...
  missed kw: ['mixed_precision', 'half_precision']

How to integrate DeepSpeed ZeRO for large-scale model traini...
  missed kw: ['stage2']

How to load a GPTQ quantized model for efficient inference?...
  missed kw: ['exllama

In [ ]:


from langchain_chroma import Chroma
from langchain_core.documents import Document


def search_with_threshold(
    vector_store: Chroma,
    query: str,
    project: str | None = None,
    k: int = 10,
    min_similarity: float = 0.20,
    fallback_k: int = 3,
) -> list[Document]:
    """
    Retrieve chunks above a similarity threshold, with a fallback.

    Strategy:
      1. Pull top-k candidates from Chroma (filtered by project if given).
      2. Convert distance -> similarity (1 - distance).
      3. Keep everything >= min_similarity.
      4. If nothing clears the threshold, fall back to top-`fallback_k`
         candidates marked with below_threshold=True so the caller knows.

    Each returned Document carries:
      - metadata["similarity"]      float in roughly [0, 1]
      - metadata["below_threshold"] True only when fallback was used
    """

    # Optional project filter — Chroma's `where` clause, exact-match metadata
    search_kwargs = {"k": k}
    if project:
        search_kwargs["filter"] = {"project": project}

    raw = vector_store.similarity_search_with_score(query, **search_kwargs)

    # Convert distance -> similarity for every candidate
    scored = []
    for doc, distance in raw:
        similarity = 1 - distance
        # Mutate a copy of metadata to avoid surprising other callers
        doc.metadata = {
            **doc.metadata,
            "similarity": similarity,
            "below_threshold": False,
        }
        scored.append((doc, similarity))

    # Sort high-to-low (Chroma usually does this, but make it explicit)
    scored.sort(key=lambda x: x[1], reverse=True)

    # Apply threshold
    passing = [doc for doc, sim in scored if sim >= min_similarity]

    if passing:
        return passing

    # Fallback: nothing cleared the bar — return top few, flagged
    fallback = []
    for doc, sim in scored[:fallback_k]:
        doc.metadata = {**doc.metadata, "below_threshold": True}
        fallback.append(doc)

    return fallback

In [ ]:
# Eval... retrieval quality with threshold

def evaluate_threshold_retrieval(
    vector_store, eval_set, project="fastapi", k=10, min_similarity=0.20
):
    results = {
        "total": len(eval_set),
        "source_hits": 0,
        "keyword_hits": 0,
        "avg_similarity": 0,
        "avg_results_returned": 0,
        "fallback_count": 0,
        "details": [],
    }

    all_similarities = []

    for test in eval_set:
        retrieved = search_with_threshold(
            vector_store, test["question"],
            project=project, k=k, min_similarity=min_similarity,
        )
        print(retrieved)

        # Check 1: Source match
        retrieved_sources = [doc.metadata["source"] for doc in retrieved]
        source_found = any(
            expected in retrieved_sources
            for expected in test["expected_sources"]
        )

        # Check 2: Keyword match
        all_text = " ".join(doc.page_content.lower() for doc in retrieved)
        keywords_found = sum(
            1 for kw in test["expected_keywords"]
            if kw.lower() in all_text
        )
        keyword_score = keywords_found / len(test["expected_keywords"])

        # Check 3: Similarity scores
        similarities = [doc.metadata.get("similarity", 0) for doc in retrieved]
        avg_sim = sum(similarities) / len(similarities) if similarities else 0
        all_similarities.extend(similarities)

        # Check 4: Did fallback trigger?
        used_fallback = any(doc.metadata.get("below_threshold") for doc in retrieved)

        results["source_hits"] += int(source_found)
        results["keyword_hits"] += keyword_score
        results["avg_results_returned"] += len(retrieved)
        results["fallback_count"] += int(used_fallback)

        results["details"].append({
            "question": test["question"],
            "source_match": source_found,
            "keyword_score": f"{keyword_score:.0%}",
            "results_returned": len(retrieved),
            "similarities": [f"{s:.4f}" for s in similarities],
            "avg_similarity": f"{avg_sim:.4f}",
            "used_fallback": used_fallback,
            "top_sources": retrieved_sources[:3],
        })

    # Summary
    total = results["total"]
    source_accuracy = results["source_hits"] / total
    keyword_accuracy = results["keyword_hits"] / total
    avg_returned = results["avg_results_returned"] / total
    avg_sim = sum(all_similarities) / len(all_similarities) if all_similarities else 0

    print(f"\n{'='*55}")
    print(f"  THRESHOLD RETRIEVAL EVALUATION (min={min_similarity})")
    print(f"{'='*55}")
    print(f"  Source accuracy:     {source_accuracy:.0%}")
    print(f"  Keyword accuracy:    {keyword_accuracy:.0%}")
    print(f"  Avg similarity:      {avg_sim:.4f}")
    print(f"  Avg results/query:   {avg_returned:.1f} / {k}")
    print(f"  Fallback triggered:  {results['fallback_count']} / {total}")
    print(f"{'='*55}\n")

    for d in results["details"]:
        status = "✓" if d["source_match"] else "✗"
        fallback = " (FALLBACK)" if d["used_fallback"] else ""
        print(f"  {status} {d['question']}{fallback}")
        print(f"    Keywords:    {d['keyword_score']}")
        print(f"    Returned:    {d['results_returned']} chunks")
        print(f"    Avg sim:     {d['avg_similarity']}")
        print(f"    Scores:      {d['similarities']}")
        print(f"    Sources:     {d['top_sources']}")
        print()

    return results


# Run at different thresholds to find the sweet spot
for threshold in [0.10, 0.15, 0.20]:
    evaluate_threshold_retrieval(
        vector_store, eval_set, min_similarity=threshold
    )

In [ ]:
# Debug: see what ChromaDB is actually returning
results_with_scores = vector_store.similarity_search_with_score(
    "What is FastAPI?", k=5, filter={"project": "fastapi"}
)

for doc, score in results_with_scores:
    print(f"  Raw score: {score:.6f}  |  1-score: {1-score:.6f}  |  {doc.metadata['source']}")

In [ ]:
eval_set = [
    # ─────────────────────────────────────────────────────────
    # TIER 1 — Easy: query phrasing closely matches doc titles
    # These should all hit rank 1 reliably.
    # ─────────────────────────────────────────────────────────
    {
        "question": "What is FastAPI?",
        "expected_sources": ["index.md", "features.md"],
        "expected_keywords": ["framework", "python", "api", "high-performance"],
    },
    {
        "question": "How to handle path parameters?",
        "expected_sources": ["path-params.md"],
        "expected_keywords": ["path", "parameter", "type"],
    },
    {
        "question": "How to use query parameters?",
        "expected_sources": ["query-params.md"],
        "expected_keywords": ["query", "parameter", "default"],
    },
    {
        "question": "How to receive a request body in FastAPI?",
        "expected_sources": ["body.md"],
        "expected_keywords": ["body", "pydantic", "model", "json"],
    },
    {
        "question": "How to deploy FastAPI with Docker?",
        "expected_sources": ["docker.md"],
        "expected_keywords": ["docker", "container", "dockerfile"],
    },
    {
        "question": "How to test a FastAPI application?",
        "expected_sources": ["testing.md"],
        "expected_keywords": ["test", "testclient", "pytest"],
    },
    {
        "question": "How to use OAuth2 with JWT in FastAPI?",
        "expected_sources": ["oauth2-jwt.md"],
        "expected_keywords": ["token", "jwt", "password", "bearer"],
    },
    {
        "question": "How to add CORS middleware?",
        "expected_sources": ["cors.md"],
        "expected_keywords": ["cors", "origin", "middleware"],
    },
    {
        "question": "How to upload files in FastAPI?",
        "expected_sources": ["request-files.md"],
        "expected_keywords": ["file", "upload", "uploadfile"],
    },
    {
        "question": "How to serve static files?",
        "expected_sources": ["static-files.md"],
        "expected_keywords": ["static", "files", "mount"],
    },

    # ─────────────────────────────────────────────────────────
    # TIER 2 — Medium: query phrasing differs from doc titles,
    # may overlap with adjacent topics
    # ─────────────────────────────────────────────────────────
    {
        "question": "How do I validate string lengths on query parameters?",
        "expected_sources": ["query-params-str-validations.md"],
        "expected_keywords": ["query", "validation", "max_length", "min_length"],
    },
    {
        "question": "How to add numeric constraints to path parameters?",
        "expected_sources": ["path-params-numeric-validations.md"],
        "expected_keywords": ["path", "numeric", "ge", "le", "gt"],
    },
    {
        "question": "How to use Pydantic Field for body validation?",
        "expected_sources": ["body-fields.md"],
        "expected_keywords": ["field", "pydantic", "validation"],
    },
    {
        "question": "How do I use nested Pydantic models in request bodies?",
        "expected_sources": ["body-nested-models.md"],
        "expected_keywords": ["nested", "model", "list", "pydantic"],
    },
    {
        "question": "How do I read cookies from a request?",
        "expected_sources": ["cookie-params.md"],
        "expected_keywords": ["cookie", "parameter"],
    },
    {
        "question": "How to read custom request headers?",
        "expected_sources": ["header-params.md"],
        "expected_keywords": ["header", "parameter"],
    },
    {
        "question": "How to handle form data submissions?",
        "expected_sources": ["request-forms.md"],
        "expected_keywords": ["form", "data"],
    },
    {
        "question": "How to return errors with status codes?",
        "expected_sources": ["handling-errors.md"],
        "expected_keywords": ["error", "httpexception", "status"],
    },
    {
        "question": "How to declare what my endpoint returns?",
        "expected_sources": ["response-model.md"],
        "expected_keywords": ["response", "model", "return"],
    },
    {
        "question": "How to change the HTTP status code of a response?",
        "expected_sources": ["response-status-code.md"],
        "expected_keywords": ["status", "code", "response"],
    },
    {
        "question": "How does dependency injection work in FastAPI?",
        "expected_sources": ["dependencies.md", "index.md"],  # may live in dependencies/index.md
        "expected_keywords": ["depends", "injection", "dependency"],
    },
    {
        "question": "How to run code in the background after returning a response?",
        "expected_sources": ["background-tasks.md"],
        "expected_keywords": ["background", "tasks", "async"],
    },
    {
        "question": "How to organize a large FastAPI project with multiple files?",
        "expected_sources": ["bigger-applications.md"],
        "expected_keywords": ["router", "apirouter", "include"],
    },
    {
        "question": "How to use a database with FastAPI?",
        "expected_sources": ["sql-databases.md"],
        "expected_keywords": ["sql", "database", "sqlmodel"],
    },
    {
        "question": "How to add custom middleware to my app?",
        "expected_sources": ["middleware.md"],
        "expected_keywords": ["middleware", "request", "response"],
    },

    # ─────────────────────────────────────────────────────────
    # TIER 3 — Harder: conceptual / paraphrased / multi-topic
    # These exercise breadcrumb embedding and re-ranker need
    # ─────────────────────────────────────────────────────────
    {
        "question": "When should I use async def vs def for route handlers?",
        "expected_sources": ["async.md"],
        "expected_keywords": ["async", "await", "concurrency"],
    },
    {
        "question": "Why does FastAPI use Python type hints?",
        "expected_sources": ["python-types.md", "features.md"],
        "expected_keywords": ["type", "hints", "annotations"],
    },
    {
        "question": "How is FastAPI different from Flask or Django?",
        "expected_sources": ["alternatives.md"],
        "expected_keywords": ["flask", "django", "comparison"],
    },
    {
        "question": "How do I get the currently logged-in user in an endpoint?",
        "expected_sources": ["get-current-user.md", "simple-oauth2.md", "oauth2-jwt.md"],
        "expected_keywords": ["current", "user", "depends", "token"],
    },
    {
        "question": "How to convert Pydantic models to JSON-compatible dicts?",
        "expected_sources": ["encoder.md"],
        "expected_keywords": ["jsonable_encoder", "json", "pydantic"],
    },
]

In [ ]:
# MRR Eval
def evaluate_retrieval(eval_set, vectorstore, k=10, threshold=0.10):
    """
    Evaluate retrieval quality with rank-aware metrics + keyword check.

    Metrics:
    - source_found:   did any expected source appear in top-k? (recall floor)
    - hit_at_1:       was the very top result correct?         (strict)
    - hit_at_3:       was a correct result in top 3?           (matches typical LLM context)
    - mrr:            Mean Reciprocal Rank — rewards high ranking
    - keyword_acc:    fraction of expected keywords found across retrieved chunks
    - avg_top_score:  average similarity of rank-1 result      (sanity check)
    """

    def source_matches(retrieved, expected_set):
        """Tolerant match: 'index.md' matches 'fastapi/index.md' and vice versa."""
        return any(
            retrieved.endswith(exp) or exp.endswith(retrieved)
            for exp in expected_set
        )

    totals = {
        "source_found": 0,
        "hit_at_1": 0,
        "hit_at_3": 0,
        "mrr": 0.0,
        "keyword_acc": 0.0,
        "avg_top_score": 0.0,
    }
    per_query = []

    for test in eval_set:
        query = test["question"]
        expected_sources = set(test["expected_sources"])
        expected_keywords = [kw.lower() for kw in test.get("expected_keywords", [])]

        # Retrieve with scores, apply loose threshold as garbage filter
        raw = vectorstore.similarity_search_with_score(query, k=k)
        filtered = [(doc, 1 - dist) for doc, dist in raw if (1 - dist) >= threshold]

        sources = [doc.metadata["source"] for doc, _ in filtered]
        top_score = filtered[0][1] if filtered else 0.0

        # Combined text from retrieved chunks for keyword check
        retrieved_text = " ".join(doc.page_content for doc, _ in filtered).lower()

        # source_found — anywhere in top-k
        found = any(source_matches(s, expected_sources) for s in sources)

        # hit@1 — top result correct
        hit1 = bool(sources) and source_matches(sources[0], expected_sources)

        # hit@3 — anything correct in top 3
        hit3 = any(source_matches(s, expected_sources) for s in sources[:3])

        # MRR — reciprocal of rank of first correct hit
        rank = next(
            (i + 1 for i, s in enumerate(sources) if source_matches(s, expected_sources)),
            None,
        )
        rr = 1 / rank if rank else 0.0

        # Keyword accuracy — fraction of expected keywords present in retrieved text
        if expected_keywords:
            kw_hits = sum(1 for kw in expected_keywords if kw in retrieved_text)
            kw_acc = kw_hits / len(expected_keywords)
        else:
            kw_acc = 0.0

        totals["source_found"] += int(found)
        totals["hit_at_1"] += int(hit1)
        totals["hit_at_3"] += int(hit3)
        totals["mrr"] += rr
        totals["keyword_acc"] += kw_acc
        totals["avg_top_score"] += top_score

        per_query.append({
            "question": query,
            "expected_sources": list(expected_sources),
            "retrieved_top3": sources[:3],
            "first_correct_rank": rank,
            "reciprocal_rank": round(rr, 3),
            "keyword_acc": round(kw_acc, 3),
            "top_score": round(top_score, 3),
        })

    n = len(eval_set)
    summary = {key: round(val / n, 3) for key, val in totals.items()}

    return {"summary": summary, "per_query": per_query}


def print_eval_report(results):
    """Pretty-print eval results, surface worst queries for debugging."""
    s = results["summary"]
    print("=" * 60)
    print("RETRIEVAL EVALUATION")
    print("=" * 60)
    print(f"  Source found (any rank):  {s['source_found']:.0%}")
    print(f"  Hit@1 (top result right): {s['hit_at_1']:.0%}")
    print(f"  Hit@3 (right in top 3):   {s['hit_at_3']:.0%}")
    print(f"  MRR (rank quality):       {s['mrr']:.3f}")
    print(f"  Keyword accuracy:         {s['keyword_acc']:.0%}")
    print(f"  Avg top-1 score:          {s['avg_top_score']:.3f}")
    print()

    # Sort worst queries first so you immediately see what to debug
    worst = sorted(results["per_query"], key=lambda x: x["reciprocal_rank"])
    print("PER-QUERY BREAKDOWN (worst first):")
    print("-" * 60)
    for q in worst:
        rank_str = q["first_correct_rank"] if q["first_correct_rank"] else "NOT FOUND"
        print(f"  Q: {q['question']}")
        print(f"     First correct rank: {rank_str}  |  RR: {q['reciprocal_rank']}  |  Top score: {q['top_score']}")
        print(f"     Keyword acc: {q['keyword_acc']:.0%}")
        print(f"     Expected: {q['expected_sources']}")
        print(f"     Got top-3: {q['retrieved_top3']}")
        print()

In [ ]:
test_set = [
    {
        "query": "What is FastAPI?",
        "expected_sources": ["fastapi/index.md", "fastapi/features.md"],
        # keep your existing "expected_keywords" if you also want keyword eval
    },
    # ... more test cases
]

results = evaluate_retrieval(eval_set, vector_store, k=10, threshold=0.3)
print_eval_report(results)

In [ ]:
all_data = vector_store._collection.get()
sources_present = sorted(set(m["source"] for m in all_data["metadatas"]))

failed_files = [
    "cookie-params.md",
    "request-forms.md",
    "response-model.md",
    "get-current-user.md",
]

print("Files in corpus:", len(sources_present))
print()
for f in failed_files:
    status = "✓ IN CORPUS" if f in sources_present else "✗ MISSING"
    print(f"  {status}: {f}")

print("\nAll sources containing 'cookie', 'form', 'response-model', or 'current-user':")
for s in sources_present:
    if any(kw in s.lower() for kw in ["cookie", "form", "response-model", "current-user", "current_user"]):
        print(f"  {s}")

In [ ]:
# Direct call to Chroma, no threshold filtering at all
query = "How do I read cookies from a request?"
raw = vector_store.similarity_search_with_score(query, k=10)

print(f"Raw results returned: {len(raw)}")
print()
for doc, dist in raw:
    sim = 1 - dist
    print(f"  dist={dist:.4f}  sim={sim:+.4f}  source={doc.metadata['source']}")